In [3]:
import pandas as pd
import numpy as np
import pickle
import os

from abc import ABC, abstractmethod

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForMaskedLM

In [4]:
df = pd.read_csv("/content/cleaned_data.csv")
df

,accession,length,gc_content,sequence,label
0,NC_036615.1,2330,40.26,GGCATAAGCAGAGGATTTTATAACAATGGAAATAAACCCATATCTA...,influenza D
1,NC_036616.1,2364,39.89,GGCATAAGCAGAGGATGTCACTACTATTAACGCTCGCAAAAGAGTA...,influenza D
2,NC_036617.1,1775,44.00,GGCATAAGCAGGAGATTATTAAGCAATATGGACTCAACAAAAGCCC...,influenza D
3,NC_036618.1,2049,43.34,AGCATAAGCAGGAGATTTTCAAAGATGTTTTTGCTTCTAGCAACAA...,influenza D
4,NC_036619.1,2195,39.54,GGCATAAGCAGGAGATTTAGAAATGTCTAGTGTAATCAGAGAAATC...,influenza D
...,...,...,...,...,...
10462,AF389122.1,890,44.72,AGCAAAAGCAGGGTGACAAAGACATAATGGATCCAAACACTGTGTC...,H1N1
10463,AF251391.1,988,47.87,TATTGAAAGATGAGCCTTCTAACCGAGGTCGAAACGTATGTTCTCT...,H3N2
10464,AF102656.1,1350,44.30,ATGAATCCAAATCAGAAGATAATAACCATTGGATCAATCTGCATGG...,H5N1
10465,J02150.1,890,44.27,AGCAAAAGCAGGGTGACAAAGACATAATGGATCCAAACACTGTGTC...,H1N1


In [5]:
X = df['sequence']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(8373,)
(8373,)
(2094,)
(2094,)


In [6]:
class UniversalModel(ABC):
    def __init__(self, name):
        self.name = name
        self.model = None

    @abstractmethod
    def train(self, X_train, y_train, **kwargs):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    def evaluate(self, X_test, y_test, target_names):
        predictions = self.predict(X_test)

        print(metrics.classification_report(y_test, predictions, target_names=target_names))
        macro_f1 = metrics.f1_score(y_test, predictions, average="macro")
        confusion_matrix = metrics.confusion_matrix(y_test, predictions)
        print(f"{macro_f1}")

        return macro_f1, confusion_matrix

In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

class HyenaDNAModel(UniversalModel):
    def __init__(self, num_labels, model_name="LongSafari/hyenadna-small-32k-seqlen-hf"):
        super().__init__(name="HyenaDNA")
        self.num_labels = num_labels
        self.label_encoder = LabelEncoder()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            trust_remote_code=True
        )
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def _tokenize(self, X):
        return self.tokenizer(
            list(X),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=32_000
        )

    def train(self, X_train, y_train, epochs=10, batch_size=16, lr=2e-5, **kwargs):
        encodings = self._tokenize(X_train)
        encoded_labels = self.label_encoder.fit_transform(y_train)
        labels = torch.tensor(encoded_labels, dtype=torch.long)

        dataset = TensorDataset(encodings["input_ids"], labels)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)

        self.model.train()
        for epoch in range(epochs):
            total_loss = 0
            for input_ids, batch_labels in loader:
                input_ids = input_ids.to(self.device)
                batch_labels = batch_labels.to(self.device)

                optimizer.zero_grad()
                outputs = self.model(input_ids=input_ids, labels=batch_labels)
                outputs.loss.backward()
                optimizer.step()
                total_loss += outputs.loss.item()

            print(f"Epoch {epoch + 1}/{epochs} — Loss: {total_loss / len(loader):.4f}")

    def predict(self, X):
        encodings = self._tokenize(X)
        dataset = TensorDataset(encodings["input_ids"])
        loader = DataLoader(dataset, batch_size=16)

        self.model.eval()
        all_preds = []
        with torch.no_grad():
            for (input_ids,) in loader:  # note the tuple unpack
                outputs = self.model(input_ids=input_ids.to(self.device))
                all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())

        return self.label_encoder.inverse_transform(all_preds)

In [8]:
target_names = list(set(y))

In [9]:
model = HyenaDNAModel(num_labels=len(target_names))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/981 [00:00<?, ?B/s]

configuration_hyena.py:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LongSafari/hyenadna-small-32k-seqlen-hf:
- configuration_hyena.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

...

modeling_hyena.py:   0%|          | 0.00/22.6k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LongSafari/hyenadna-small-32k-seqlen-hf:
- modeling_hyena.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

[transformers] HyenaDNAForSequenceClassification LOAD REPORT from: LongSafari/hyenadna-small-32k-seqlen-hf
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
model.train(X_train, y_train, epochs=30)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch 1/30 — Loss: 1.9006
Epoch 2/30 — Loss: 1.1684
Epoch 3/30 — Loss: 0.8781
Epoch 4/30 — Loss: 0.7213
Epoch 5/30 — Loss: 0.6117
Epoch 6/30 — Loss: 0.5299
Epoch 7/30 — Loss: 0.4642
Epoch 8/30 — Loss: 0.4086
Epoch 9/30 — Loss: 0.3714
Epoch 10/30 — Loss: 0.3421
Epoch 11/30 — Loss: 0.3052
Epoch 12/30 — Loss: 0.2808
Epoch 13/30 — Loss: 0.2559
Epoch 14/30 — Loss: 0.2429
Epoch 15/30 — Loss: 0.2314
Epoch 16/30 — Loss: 0.2077
Epoch 17/30 — Loss: 0.1946
Epoch 18/30 — Loss: 0.1815
Epoch 19/30 — Loss: 0.1786
Epoch 20/30 — Loss: 0.1607
Epoch 21/30 — Loss: 0.1581
Epoch 22/30 — Loss: 0.1554
Epoch 23/30 — Loss: 0.1508
Epoch 24/30 — Loss: 0.1374
Epoch 25/30 — Loss: 0.1355
Epoch 26/30 — Loss: 0.1264
Epoch 27/30 — Loss: 0.1379
Epoch 28/30 — Loss: 0.1138
Epoch 29/30 — Loss: 0.1118
Epoch 30/30 — Loss: 0.1062


In [12]:
model.evaluate(X_test, y_test, target_names=target_names)

              precision    recall  f1-score   support

        H4N6       0.90      0.91      0.90       494
        H3N2       0.61      0.49      0.54        80
        H7N9       0.86      0.90      0.88       394
        H1N2       0.74      0.75      0.74        56
        H5N2       0.59      0.65      0.62        26
        H5N6       0.95      0.93      0.94       227
        H3N8       0.94      0.85      0.89       156
        H6N2       0.77      0.83      0.80        36
        H7N2       0.90      0.93      0.91        40
        H7N3       0.38      0.30      0.33        20
 influenza B       0.63      0.55      0.59        22
 influenza D       0.74      0.82      0.78       121
        H9N2       0.46      0.38      0.41        32
        H5N1       0.81      0.83      0.82        84
        H5N8       0.88      0.90      0.89       220
        H1N1       0.98      0.96      0.97        56
        H7N7       0.94      0.97      0.95        30

    accuracy              

(0.7633711521769365,
 array([[449,  17,  26,   0,   2,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0],
        [ 16,  39,  23,   0,   1,   0,   1,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0],
        [ 28,   7, 353,   2,   2,   0,   0,   0,   1,   1,   0,   0,   0,
           0,   0,   0,   0],
        [  0,   0,   2,  42,   2,   1,   1,   0,   1,   1,   0,   3,   1,
           0,   1,   1,   0],
        [  1,   0,   2,   0,  17,   0,   0,   1,   0,   1,   0,   1,   0,
           0,   3,   0,   0],
        [  0,   1,   0,   0,   1, 211,   0,   7,   0,   1,   0,   2,   1,
           1,   2,   0,   0],
        [  0,   0,   0,   4,   0,   1, 133,   0,   2,   2,   2,   4,   0,
           0,   7,   0,   1],
        [  0,   0,   0,   0,   0,   4,   1,  30,   0,   0,   0,   0,   0,
           0,   1,   0,   0],
        [  0,   0,   0,   0,   0,   0,   2,   0,  37,   0,   0,   0,   0,
           0,   1,   0,   0],
        [  1,   0,   2,   1,   0,   2,   1,

In [17]:
def save_model(model, path):
    os.makedirs(path, exist_ok=True)
    torch.save(model.model.state_dict(), os.path.join(path, "model.pt"))
    model.tokenizer.save_pretrained(path)
    with open(os.path.join(path, "label_encoder.pkl"), "wb") as f:
        pickle.dump(model.label_encoder, f)
    print(f"Saved to {path}")

def load_model(path, num_labels):
    m = HyenaDNAModel(num_labels=num_labels)
    m.model.load_state_dict(torch.load(os.path.join(path, "model.pt"), map_location=m.device))
    m.tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    with open(os.path.join(path, "label_encoder.pkl"), "rb") as f:
        m.label_encoder = pickle.load(f)
    return m

In [18]:
save_model(model, 'hyenadna')

Saved to hyenadna


In [19]:
hyenadna = load_model('hyenadna', len(target_names))

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

[transformers] HyenaDNAForSequenceClassification LOAD REPORT from: LongSafari/hyenadna-small-32k-seqlen-hf
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
!zip -r hyenadna.zip hyenadna/

  adding: hyenadna/ (stored 0%)
  adding: hyenadna/model.pt (deflated 14%)
  adding: hyenadna/label_encoder.pkl (deflated 29%)
  adding: hyenadna/configuration_hyena.py (deflated 74%)
  adding: hyenadna/config.json (deflated 63%)
  adding: hyenadna/tokenizer_config.json (deflated 76%)
  adding: hyenadna/modeling_hyena.py (deflated 74%)
  adding: hyenadna/tokenization_hyena.py (deflated 68%)
